In [ ]:
import anndata as ad
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import warnings
import os  
import loompy
import gzip
import shutil
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(dpi=50, dpi_save=300, format='png', 
                             frameon=False, transparent=True, fontsize=10, figsize=(4, 4))

warnings.simplefilter(action='ignore', category=FutureWarning)

plt.rcParams["image.aspect"] = "equal"
plt.rcParams["figure.figsize"] = ([4, 4])  
mpl.rcParams['pdf.fonttype'] = 42

colorrs = ["#4DBBD5", "#00A087", "#E64B35","#3C5488", "#F39B7F", "#8491B4",
        "#91D1C2",  "#B9C984",  "#9ACBDE", "#F494BE", "#EDCAE0", 
        "#C8CADF", "#F47892", "#F6A395",  "#C9AFA2", "#ABADC5", "#AEB9AC", 
        "#4b6aa8", "#3ca0cf", "#c376a7", "#ad98c3", "#cea5c7",
        "#53738c", "#a5a9b0", "#a78982", "#696a6c", "#92699e",
        "#d69971", "#df5734", "#6c408e", "#ac6894", "#d4c2db",
        "#537eb7", "#83ab8e", "#ece399", "#405993", "#cc7f73",
        "#b95055", "#d5bb72", "#bc9a7f", "#e0cfda", "#d8a0c0",
        "#d69a55", "#64a776", "#cbdaa9",
        "#efd2c9", "#da6f6d", "#ebb1a4", "#a44e89", "#a9c2cb",
        "#b85292", "#6d6fa0", "#8d689d", "#c8c7e1", "#d25774",
        "#c49abc", "#927c9a", "#3674a2", "#9f8d89", "#72567a",
        "#63a3b8", "#c4daec", "#61bada", "#b7deea", "#e29eaf",
        "#4490c4", "#e6e2a3",  "#c4612f", "#9a70a8",
        "#76a2be", "#408444", "#c6adb0", "#9d3b62", "#2d3462"]

In [ ]:
sample_info_path = "/home/xiaoquan/scanpy/KP/CellRanger/sample.csv"
sample_info = pd.read_csv(sample_info_path)

sample_dirs = [
    "M01", "M02", "M03", "M04", "M05", "M06", "M07", "M08", "M09", "M10",
    "M11", "M12", "M13", "M14", "M15", "M16", "M17", "M18", "M19", "M20",
    "M21", "M22", "M23", "S01", "S02", "S03", "S04", "S05", "S06", "S07",
    "S08", "S09", "S10", "S11", "S12", "S13", "S14", "S15", "S16", "S17",
    "S18", "S19", "S20", "S21", "S22", "S23", "HC01", "HC02", "HC03", "HC04",
    "HC05", "HC06", "HC07", "HC08", "HC09", "HC10", "HC11", "HC12", "HC13", "HC14",
    "HC15", "HC16", "HC17", "HC18", "HC19", "HC20", "HC21", "HC22", "HC23", "HC24",
    "HC25", "HC26", "HC27", "HC28", "HC29", "HC30", "HC31", "HC32", "HC33", "HC34",
    "HC35", "HC36", "HC37", "HC38", "HC39", "HC40", "HC41", "HC42", "HC43", "HC44",
    "HC45", "HC46", "HC47", "HC48", "HC49", "HC50", "HC51", "HC52", "HC53", "HC54"
]

base_path = "/home/xiaoquan/scanpy/KP/CellRanger"
adatas = []
for sd in sample_dirs:
    path = os.path.join(base_path, sd)
    ad = sc.read_10x_mtx(path, var_names='gene_symbols')
    ad.obs["Sample"] = sd
    if sd.startswith("HC"):
        group = "Healthy Controls"
        condition = "HC"
    elif sd.startswith("M"):
        group = "Mild Klebsiella pneumoniae pneumonia" 
        condition = "MKPP"
    else:
        group = "Severe Klebsiella pneumoniae pneumonia"
        condition = "SKPP"
    
    ad.obs["Group"] = group
    ad.obs["Condition"] = condition 
    adatas.append(ad)

adata = sc.concat(adatas, label='Sample',keys=sample_dirs, index_unique='-',join="inner")

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata

In [ ]:
import scrublet as scr

In [ ]:
sc.external.pp.scrublet(adata, expected_doublet_rate=0.05, threshold=0.25, batch_key="Sample")

In [ ]:
adata = adata[adata.obs["predicted_doublet"] == False]

In [ ]:
def basic_qc(adata):

    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    mt_gene = adata.var.index[adata.var_names.str.startswith('MT-')]
    
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var['rp'] = [i.startswith('RPL') or i.startswith('RPS') for i in adata.var_names]
    sc.pp.calculate_qc_metrics(adata, qc_vars=['rp'], percent_top=None, log1p=False, inplace=True)
    rp_gene = [i for i in adata.var_names if i.startswith('RPL') or i.startswith('RPS')]
    
    adata.var['hb'] = [i.startswith('HB') and not i.startswith('HBP') for i in adata.var_names]
    sc.pp.calculate_qc_metrics(adata, qc_vars=['hb'], percent_top=None, log1p=False, inplace=True)
    hb_gene = [i for i in adata.var_names if i.startswith('HB') and not i.startswith('HBP')]
     adata = adata[adata.obs.n_genes_by_counts < 5000, :]
    adata = adata[adata.obs.n_genes_by_counts > 200, :] 
    adata = adata[adata.obs.pct_counts_mt < 10, :]
    adata = adata[adata.obs.pct_counts_rp > 3, :]
    adata = adata[adata.obs.pct_counts_hb < 1, :]   
    return adata

In [ ]:
adata = basic_qc(adata)

In [ ]:
import harmonypy as hm 

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata
adata.layers["log1p_norm"] = adata.X.copy()

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["ig"] = adata.var_names.str.startswith(("IGH", "IGK", "IGL"))

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, n_top_genes=1500)
adata.var["hvg_original"] = adata.var["highly_variable"].copy()
adata.var["hvg_for_pca"] = (
    adata.var["highly_variable"]
    & ~adata.var["mt"]
    & ~adata.var["rp"]
    & ~adata.var["ig"]
)
adata.var["highly_variable"] = adata.var["hvg_for_pca"].copy()

In [ ]:
adata.X = adata.layers["log1p_norm"].copy()
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)
sc.tl.pca(
    adata,
    svd_solver="arpack",
    use_highly_variable=True
)

In [ ]:
sc.external.pp.harmony_integrate(adata, 
                                key=['Sample', 'Dataset'],  
                                theta=[2.5, 1.5], basis='X_pca', adjusted_basis='X_pca_harmony')  

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color='Sample', palette=colorrs, title='After Harmony', legend_loc=None)

In [ ]:
resolutions = [0.5, 1.0, 1.5, 2.0]
for res in resolutions:
    sc.tl.louvain(adata, resolution=res, key_added=f'louvain_{res}')

In [ ]:
for k in ["louvain_0.5", "louvain_1.0", "louvain_1.5", "louvain_2.0", "Group"]:
    sc.pl.umap(adata,color=k, palette=colorr)

In [ ]:
sc.tl.rank_genes_groups(
    T_cell,
    groupby="louvain_2.0",
    method="wilcoxon",
    use_raw=True,
    pts=True,
    key_added="rank_genes_groups"
)

In [ ]:
markers = ["CD79A", "CD79B","MS4A1",
           "MZB1",  "IGHG1", 
           "CD3D", "CD3E", "CD40LG", 
           "CD8A", "CD8B",
           "TRDV2", "TRGV9", 
           "SLC4A10", "TRAV1-2", 
           "KLRF1", "NKG7", "TYROBP", 
          "CST3", "LYZ", "CD14","FCGR3A","CD1C", "ITGAX",
          "PPBP", "PF4"]
sc.pl.dotplot(adata, var_names=markers, groupby='louvain_2.0', ax=ax, 
              standard_scale='var', use_raw=True, show=False)

In [ ]:
condition_order = ["HC", "MKPP", "SKPP"]
condition_labels = ["Healthy controls", "Mild symptoms", "Severe symptoms"]
condition_colors = {
    "HC": "#4DBBD5B2", 
    "MKPP": "#00A087B2",  
    "SKPP": "#E64B35B2"
}

x_min, x_max = np.min(adata.obsm['X_umap'][:, 0]), np.max(adata.obsm['X_umap'][:, 0])
y_min, y_max = np.min(adata.obsm['X_umap'][:, 1]), np.max(adata.obsm['X_umap'][:, 1])

fig, axes = plt.subplots(1, len(condition_order), figsize=(3 * len(condition_order), 3.5), 
                        dpi=300, sharex=True, sharey=True,
                        gridspec_kw={'wspace': 0, 'hspace': 0})  

for i, condition in enumerate(condition_order):
    ax = axes[i]

    adata_bg = adata[adata.obs['Condition'] != condition]
    ax.scatter(
        adata_bg.obsm['X_umap'][:, 0],
        adata_bg.obsm['X_umap'][:, 1],
        s=0.01,
        color="lightgray",
        rasterized=True
    )

    adata_fg = adata[adata.obs['Condition'] == condition]
    ax.scatter(
        adata_fg.obsm['X_umap'][:, 0],
        adata_fg.obsm['X_umap'][:, 1],
        s=0.005,
        alpha=0.7,
        color=condition_colors[condition],
        rasterized=True
    )
    
    ax.set_title(condition_labels[i], fontsize=13, fontweight='bold')
    ax.set_facecolor("white")
    ax.grid(False)
    
    if i == 0: 
        ax.spines['left'].set_visible(True)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=True, labelleft=True, right=False)
    elif i == len(condition_order) - 1: 
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=False, right=False, labelright=False)
    else: 
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=False, right=False)

    ax.tick_params(bottom=False, labelbottom=False)

for ax in axes:
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

axes[0].set_ylabel('UMAP2', fontsize=12)
axes[1].set_xlabel('UMAP1', fontsize=12)

plt.show()

In [ ]:
count_df = adata.obs.groupby(["Condition", "celltype_major"]).size().reset_index(name="count")
count_df["Proportion"] = count_df.groupby(["Condition"])["count"].transform(lambda x: x/x.sum())

celltype_order = ['B', 'Plasma', 'CD4T', 'CD8T', 'γδT', 'MAIT', 'DNT', 'NK', 'Mono', 'DC', 'Mega']
count_df['celltype_major'] = pd.Categorical(count_df['celltype_major'], categories=celltype_order, ordered=True)
count_df = count_df.sort_values(['Condition', 'celltype_major'])

conditions = count_df['Condition'].unique()
n_groups = len(conditions)
colors = dict(zip(celltype_order, colorrs)) 

fig, axes = plt.subplots(1, n_groups, figsize=(n_groups * 3, 3), subplot_kw={'projection': 'polar'})

if n_groups == 1:
    axes = [axes]

for ax, cond in zip(axes, conditions):
    data = count_df[count_df['Condition'] == cond]

    ratios = data['Proportion'].values
    angles = np.concatenate(([0], np.cumsum(ratios) * 2 * np.pi))
    
     inner_radius = 1.5 
    outer_radius = 5 
    width = outer_radius - inner_radius
    
    for i, celltype in enumerate(data['celltype_major']):
        start_angle = angles[i]
        end_angle = angles[i+1]
        
        ax.bar(x=(start_angle + end_angle) / 2, 
               height=width, 
               width=end_angle - start_angle, 
               bottom=inner_radius,
               color=colors[celltype],
               edgecolor='white', 
               linewidth=1.0,
               alpha=1.0)


    ax.set_axis_off()
    ax.set_title(cond, fontsize=14, fontweight='bold', pad=5) 
plt.subplots_adjust(wspace=0.1)
handles = [plt.Rectangle((0,0),1,1, color=colors[ct]) for ct in celltype_order if ct in count_df['celltype_major'].unique()]
labels = [ct for ct in celltype_order if ct in count_df['celltype_major'].unique()]
fig.legend(handles, labels, title='Celltypes', bbox_to_anchor=(1.0, 0.5), loc='center left', fontsize=12, title_fontsize=14)

plt.show()

<h1>Result_2</h1>

In [ ]:
inflammatory_genes = [
    "ABCA1", "ABI1", "ACVR1B", "ACVR2A", "ADGRE1", "ADM", "ADORA2B",
    "ADRM1", "AHR", "APLNR", "AQP9", "ATP2A2", "ATP2B1", "ATP2C1",
    "AXL", "BDKRB1", "BEST1", "BST2", "BTG2", "C3AR1", "C5AR1",
    "CALCRL", "CCL17", "CCL2", "CCL20", "CCL22", "CCL24", "CCL5",
    "CCL7", "CCR7", "CCRL2", "CD14", "CD40", "CD48", "CD55", "CD69",
    "CD70", "CD82", "CDKN1A", "CHST2", "CLEC5A", "CMKLR1", "CSF1",
    "CSF3", "CSF3R", "CX3CL1", "CXCL10", "CXCL11", "CXCL6", "CXCL8",
    "CXCL9", "CXCR6", "CYBB", "DCBLD2", "EBI3", "EDN1", "EIF2AK2",
    "EMP3", "EREG", "F3", "FFAR2", "FPR1", "FZD5", "GABBR1", "GCH1",
    "GNA15", "GNAI3", "GP1BA", "GPC3", "GPR132", "GPR183", "HAS2",
    "HBEGF", "HIF1A", "HPN", "HRH1", "ICAM1", "ICAM4", "ICOSLG",
    "IFITM1", "IFNAR1", "IFNGR2", "IL10", "IL10RA", "IL12B", "IL15",
    "IL15RA", "IL18", "IL18R1", "IL18RAP", "IL1A", "IL1B", "IL1R1",
    "IL2RB", "IL4R", "IL6", "IL7R", "INHBA", "IRAK2", "IRF1", "IRF7",
    "ITGA5", "ITGB3", "ITGB8", "KCNA3", "KCNJ2", "KCNMB2", "KIF1B",
    "KLF6", "LAMP3", "LCK", "LCP2", "LDLR", "LIF", "LPAR1", "LTA",
    "LY6E", "LYN", "MARCO", "MEFV", "MEP1A", "MET", "MMP14", "MSR1",
    "MXD1", "MYC", "NAMPT", "NDP", "NFKB1", "NFKBIA", "NLRP3", "NMI",
    "NMUR1", "NOD2", "NPFFR2", "OLR1", "OPRK1", "OSM", "OSMR", "P2RX4",
    "P2RX7", "P2RY2", "PCDH7", "PDE4B", "PDPN", "PIK3R5", "PLAUR",
    "PROK2", "PSEN1", "PTAFR", "PTGER2", "PTGER4", "PTGIR", "PTPRE",
    "PVR", "RAF1", "RASGRP1", "RELA", "RGS1", "RGS16", "RHOG", "RIPK2",
    "RNF144B", "ROS1", "RTP4", "SCARF1", "SCN1B", "SELE", "SELENOS",
    "SELL", "SEMA4D", "SERPINE1", "SGMS2", "SLAMF1", "SLC11A2",
    "SLC1A2", "SLC28A2", "SLC31A1", "SLC31A2", "SLC4A4", "SLC7A1",
    "SLC7A2", "SPHK1", "SRI", "STAB1", "TACR1", "TACR3", "TAPBP",
    "TIMP1", "TLR1", "TLR2", "TLR3", "TNFAIP6", "TNFSF14", "TNFRSF1B",
    "TNFRSF9", "TNFSF10", "TNFSF15", "TNFSF9", "TPBG", "VIP"
]
cytokine_genes = [
    "IL2", "IL7", "CSF3", "CXCL10", "CCL2", "CCL3", "TNF", "IL6",
    "CCL7", "IL1RN", "CSF1", "IFNG", "IL2RA", "IL10", "IL18", "HGF",
    "CXCL9", "CCL27", "TGFB1", "IL1B", "LTA", "CSF2", "LTB", "TNFSF13",
    "IL4", "CCL12", "CXCL8", "CXCL11", "CCL4", "CXCL1", "CXCL2", "CXCL3",
    "CCL3L1", "CCL8", "CXCL16", "IFNA1", "CCL5", "CCL11", "IFNA2",
    "CCL20", "CCL4L2", "OSM", "TNFSF14", "S100A12", "FGF19", "CXCL5",
    "CCL19", "IL18R1", "TGFA", "IFNB1", "IL8", "IL17C", "TNFSF10",
    "FGF7", "XCL1", "FGF13", "LIF", "TGFB3", "INHBE", "CERS1", "TXLNA",
    "IFNW1", "IL22", "XCL2", "CCL25", "CCL16", "CD40LG", "IL20", "FASLG",
    "TPO", "SCYL3", "PF4V1", "TNFSF8", "GDF15", "IL1A", "VEGFA", "GDF7",
    "BMP6", "PDGFA", "IL21", "ABCD-1", "ABCD-2", "PDGFB", "TNFSF4",
    "FAM19A1", "HBEGF", "PDGFD", "IL12RB2", "GH1", "VEGFB", "MIP3B",
    "IL27", "PF4", "BMP8B", "TNFSF12", "IL15", "SCYL2", "SCYL1", "TSLP",
    "GDF11", "SDF1B", "INHBA", "PPBP", "FGF11", "FGF22", "VEGFC",
    "CCL18", "TNFSF11", "IL12A", "EBI3", "AMH", "IL26", "IL32", "PDGFC",
    "FGF23", "IGF1", "IL33", "CCL28", "CLCF1", "TNFSF9", "BMP3", "IL24",
    "GDF10", "CXCL6", "GDF9", "IL23A", "IL16", "CD70", "IL5", "FGF9",
    "IFNL1", "TSC1", "FGF2", "IL23R", "IL1G", "SPP1", "IL12RB1", "BMP4",
    "IL13", "TGFB2", "TAFA2", "EDA", "MIF", "TNFSF13B", "BMP7", "FGF18",
    "CCL23", "S100A12", "S100A8", "S100A9"
]
sc.tl.score_genes(adata, gene_list=inflammatory_genes, score_name='inflammatory_score')
sc.tl.score_genes(adata, gene_list=cytokine_genes, score_name='cytokine_score')

In [ ]:
S100A8_A9_A12_genes = [
    'S100A8', 'S100A9', 'S100A12'
]   
sc.tl.score_genes(adata, gene_list=S100A8_A9_A12_genes, score_name='S100A8_A9_A12_score')

In [ ]:
skpp = adata[adata.obs["Condition"] == "SKPP"].copy()
expr = skpp.raw if skpp.raw is not None else skpp
genes = [g for g in cytokine_genes if g in expr.var_names]

contributions = np.asarray(expr[:, genes].X.sum(axis=0)).ravel()
plot_df = pd.DataFrame({"Cytokine": genes, "Contribution": contributions})
plot_df["Percentage"] = plot_df["Contribution"] / plot_df["Contribution"].sum() * 100
plot_df = plot_df.sort_values("Percentage", ascending=False)

top10 = plot_df.head(10).copy()
others = pd.DataFrame({"Cytokine": ["Others"], "Contribution": [plot_df.iloc[10:]["Contribution"].sum()],
                       "Percentage": [plot_df.iloc[10:]["Percentage"].sum()]})
plot_df = pd.concat([top10, others], ignore_index=True)

print(f"Actual cytokine_score sum: {skpp.obs['cytokine_score'].sum():.2f}")
print(f"Gene expression contribution sum: {contributions.sum():.2f}")

x = np.arange(len(plot_df))
y = plot_df["Percentage"].to_numpy()
colors = np.where(x < 4, "#F39B7F", np.where(x < 10, "#4791C1", "#A0CDE4"))

fig, ax = plt.subplots(figsize=(5, 2.5))
ax.axvspan(-0.5, 3.5, color="#FCE6DF", alpha=0.95, zorder=0)
ax.axvspan(3.5, 9.5, color="#D4F0FB", alpha=0.85, zorder=0)
ax.axvspan(9.5, 10.5, color="#F0F7FD", alpha=0.95, zorder=0)

ax.vlines(x, 0, y, colors=colors, linewidth=1.6, alpha=0.9, zorder=2)
ax.scatter(x, y, s=120, facecolors="white", edgecolors=colors, linewidths=1.6, zorder=3)
ax.scatter(x, y, s=55, c=colors, edgecolors="white", linewidths=0.4, zorder=4)

for xi, yi in zip(x, y):
    ax.text(xi, yi + y.max() * 0.035, f"{yi:.1f}%", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
labels = ax.set_xticklabels(plot_df["Cytokine"], fontsize=8, rotation=45, ha="right")
plt.setp(labels[:4], color="#E64B35")

ax.set_ylabel("Cytokine contribution (%)", fontsize=10)
ax.set_title("Relative inflammatory contribution of top 10 cytokines in SKPP", fontsize=10, fontweight="bold")
ax.set_xlim(-0.5, len(plot_df) - 0.5)
ax.set_ylim(0, y.max() * 1.22)
ax.tick_params(axis="both", labelsize=8, width=0.7, length=3)
ax.grid(False)
plt.show()

In [ ]:
cmap = LinearSegmentedColormap.from_list("cmap", ["#4DBBD5", "white", "#E64B35"])
disease_order = ["HC", "MKPP", "SKPP"]
disease_labels = ["Healthy controls", "Mild symptoms", "Severe symptoms"]

inflammatory_scores = adata.obs["inflammatory_score"]
cytokine_scores = adata.obs["cytokine_score"]
vmin_inflammatory, vmax_inflammatory = np.percentile(inflammatory_scores, [10, 90])
vmin_cytokine, vmax_cytokine = np.percentile(cytokine_scores, [10, 90])

umap = adata.obsm["X_umap"]
x_min, x_max = umap[:, 0].min(), umap[:, 0].max()
y_min, y_max = umap[:, 1].min(), umap[:, 1].max()

fig, axes = plt.subplots(2, len(disease_order), figsize=(1.5 * len(disease_order), 2.5), dpi=300,
                         sharex=True, sharey=True, gridspec_kw={"wspace": 0.1, "hspace": 0.3})

for i, disease in enumerate(disease_order):
    disease_adata = adata[adata.obs["Condition"] == disease]
    scatter1 = axes[0, i].scatter(disease_adata.obsm["X_umap"][:, 0], disease_adata.obsm["X_umap"][:, 1],
                                  c=disease_adata.obs["inflammatory_score"], cmap=cmap, s=0.02, alpha=1,
                                  vmin=vmin_inflammatory, vmax=vmax_inflammatory, edgecolor="none", rasterized=True)
    scatter2 = axes[1, i].scatter(disease_adata.obsm["X_umap"][:, 0], disease_adata.obsm["X_umap"][:, 1],
                                  c=disease_adata.obs["cytokine_score"], cmap=cmap, s=0.02, alpha=1,
                                  vmin=vmin_cytokine, vmax=vmax_cytokine, edgecolor="none", rasterized=True)
    axes[0, i].text(0.5, 1.15, disease_labels[i], transform=axes[0, i].transAxes,
                    ha="center", va="bottom", fontsize=6, fontweight="bold")

for ax in axes.flat:
    ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max))
    ax.set_facecolor("white")
    ax.grid(False)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[:, 0]:
    ax.spines["left"].set_visible(True)
    ax.spines["left"].set_linewidth(1)
    ax.set_ylabel("UMAP2", fontsize=6)

for ax in axes.flat:
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_linewidth(1)

for ax in axes[1, :]:
    ax.set_xlabel("UMAP1", fontsize=6)

fig.text(0.5, 1.0, "Inflammatory Score", ha="center", va="bottom", fontsize=6, fontweight="bold")
fig.text(0.5, 0.5, "Cytokine Score", ha="center", va="bottom", fontsize=6, fontweight="bold")

cbar1 = fig.colorbar(scatter1, ax=axes[0, :], location="right", shrink=0.6, aspect=20, pad=0.02)
cbar2 = fig.colorbar(scatter2, ax=axes[1, :], location="right", shrink=0.6, aspect=20, pad=0.02)
for cbar in [cbar1, cbar2]:
    cbar.ax.tick_params(labelsize=5)
    cbar.outline.set_visible(False)

plt.show()

In [ ]:
target_genes = [
    'S100A9', 'S100A8', 'S100A12', 'TNFSF12', 'TNFSF10',
    'FASLG',  'IL15', 'XCL2',  'HGF', 'IL7', 'CXCL1',
     'PDGFC', 'IL18',  'CXCL2'
]

colors = [ "#4DBBD5",'#F0F0F0', "#E64B35"]
custom_cmap = LinearSegmentedColormap.from_list('custom_blue_red', colors, N=100)

dotplot = sc.pl.dotplot(
    adata,
    var_names=target_genes,
    groupby='celltype',
    standard_scale='var',
    use_raw=True,
    show=False,
    color_map=custom_cmap,
    dot_max=0.8,
    smallest_dot=15,
    size_title='Fraction of cells (%)',
    colorbar_title='Mean expression',
    var_group_rotation=0,
    dendrogram=False,
    swap_axes=True,  
    figsize=(14, 4)
)

ax = dotplot['mainplot_ax']

for spine in ['left', 'bottom']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(0.8)
    ax.spines[spine].set_color('black')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

ax.set_xlabel('', fontsize=9, fontweight='bold', labelpad=10)
ax.set_ylabel('', fontsize=9, fontweight='bold', labelpad=10)
ax.tick_params(axis='x', which='major', labelsize=9, rotation=90)
ax.tick_params(axis='y', which='major', labelsize=9)
for label in ax.get_xticklabels():
    label.set_horizontalalignment('center')
    label.set_verticalalignment('top')
    
plt.show()

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list("custom_blue_red", ["#4DBBD5", "#F0F0F0", "#E64B35"], N=100)
name_map = {
    "Mono_01_Classical_CD14": "Mono_01_Classical",
    "Mono_03_Classical_CD83_HLA-DPB1": "Mono_03_Classical",
    "Mono_04_Intermedate_CD14_CD16": "Mono_04_Intermedate",
    "Mono_05_Intermedate_CD83_HLA-DPB1": "Mono_05_Intermedate",
    "Mono_06_Non-classical_CD16_C1QA": "Mono_06_Non-classical",
    "Mono_07_MDSC_CD24_ARHGAP26": "Mono_07_MDSC",
    "Mono_08_MDSC_S100A8_S100A9": "Mono_08_MDSC",
    "Mega": "Mega",
    "CD4T_07_Th1_CXCR3_ISG20": "CD4T_07_Th1",
    "NK_04_CD56_LAG3": "NK_04_CD56"
}

dotplot = sc.pl.dotplot(adata_combined, var_names=valid_genes, groupby="plot_group", standard_scale="var",
                        use_raw=False, show=False, color_map=custom_cmap, dot_max=0.8,
                        smallest_dot=15, swap_axes=True, dendrogram=False, figsize=(5, 6))

ax = dotplot["mainplot_ax"]
fig = ax.figure
ax.set_position([0.12, 0.25, 0.63, 0.65])

num_ct, num_cond, gap = len(highlight_celltypes), len(condition_order), 0.4
ticks = ax.get_xticks()
labels = [name_map.get(x.get_text(), x.get_text()) for x in ax.get_xticklabels()]
new_ticks = np.array(ticks)
new_ticks[num_ct:] += gap

offsets = ax.collections[0].get_offsets().copy()
offsets[offsets[:, 0] > ticks[num_ct - 1] + 0.1, 0] += gap
ax.collections[0].set_offsets(offsets)
ax.set_xticks(new_ticks)
ax.set_xticklabels(labels)

ct_colors = ["#B2182B", "#CB4642", "#E16F57", "#EF9674", "#FAD2BB", "#D8E3E7", "#A4CCE2", "#71ACD4", "#4A8DC4", "#337AB7"]
cond_colors = ["#4DBBD5", "#00A087", "#E64B35"]
y_bar = len(valid_genes) - 0.35

for x, color in zip(new_ticks, ct_colors + cond_colors):
    ax.add_patch(patches.Rectangle((x - 0.5, y_bar), 1, 0.5, facecolor=color,
                                   edgecolor="black", linewidth=0.3, clip_on=False))

x_ct_start, x_cond_start = new_ticks[0] - 0.5, new_ticks[num_ct] - 0.5
x_ct_width = new_ticks[num_ct - 1] - new_ticks[0] + 1
x_cond_width = new_ticks[-1] - new_ticks[num_ct] + 1
box_height = len(valid_genes) + 0.65

for x, width in [(x_ct_start, x_ct_width), (x_cond_start, x_cond_width)]:
    ax.add_patch(patches.Rectangle((x, -0.5), width, box_height, fill=False,
                                   edgecolor="black", linewidth=0.5, clip_on=False, zorder=5))

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", which="both", bottom=False, top=False, labelsize=10, pad=0)
ax.tick_params(axis="y", which="both", left=True, right=False, labelsize=10, pad=2, length=3, direction="out")
plt.setp(ax.get_xticklabels(), rotation=55, ha="right", rotation_mode="anchor")

for y in range(len(valid_genes)):
    ax.plot([x_cond_start - 0.08, x_cond_start], [y, y], color="black", linewidth=0.8, clip_on=False, zorder=6)

ax.set_xlim(x_ct_start, x_cond_start + x_cond_width)
ax.set_ylim(-0.5, len(valid_genes) + 0.2)
ax.text(x_ct_start + x_ct_width / 2, len(valid_genes) + 0.4, "Celltype", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.text(x_cond_start + x_cond_width / 2, len(valid_genes) + 0.4, "Condition", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")

dotplot["color_legend_ax"].remove()
dotplot["size_legend_ax"].remove()

cbar_ax = fig.add_axes([0.88, 0.55, 0.02, 0.12])
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(-1, 1))
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.ax.tick_params(labelsize=9)
cbar.outline.set_linewidth(0.8)
fig.text(0.82, 0.69, "Pct. exp.", fontsize=10, ha="left")

size_ax = fig.add_axes([0.88, 0.35, 0.05, 0.15])
size_ax.axis("off")
fig.text(0.82, 0.52, "Avg. exp.", fontsize=10, ha="left")

fracs = np.array([10, 30, 50])
ys = np.linspace(0.1, 0.8, len(fracs))
size_ax.scatter(np.zeros(len(fracs)), ys, s=10 + fracs, color="black")
for y, f in zip(ys, fracs):
    size_ax.text(0.6, y, str(f), ha="left", va="center", fontsize=9)

size_ax.set_xlim(-0.5, 1.5)
size_ax.set_ylim(0, 1)

plt.show()

In [ ]:
main_colors = [c[:7] for c in colorrs]
highlight_palette = {ct: main_colors[i] for i, ct in enumerate(highlight_celltypes)}
palette = {ct: highlight_palette.get(ct, "lightgray") for ct in adata.obs["celltype"].cat.categories}

fig, ax = plt.subplots(figsize=(4.8, 2.5), dpi=300)
sc.pl.umap(adata, color="celltype", size=0.5, palette=palette, ax=ax, show=False,
           title="Inflammation celltypes", frameon=True, legend_loc=None)

handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=highlight_palette[ct],
                  markersize=6, markeredgecolor="none", label=ct) for ct in highlight_celltypes]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 0.5), loc="center left", frameon=False, fontsize=8)

ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("UMAP1", fontsize=8)
ax.set_ylabel("UMAP2", fontsize=8)
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)

plt.show()

In [ ]:
cmap = LinearSegmentedColormap.from_list("cmap", ["#4DBBD5", "white", "#E64B35"])
disease_order = ["HC", "MKPP", "SKPP"]
disease_labels = ["Healthy controls", "Mild symptoms", "Severe symptoms"]

inflammatory_scores = adata.obs["inflammatory_score"]
cytokine_scores = adata.obs["cytokine_score"]
vmin_inflammatory, vmax_inflammatory = np.percentile(inflammatory_scores, [10, 90])
vmin_cytokine, vmax_cytokine = np.percentile(cytokine_scores, [10, 90])

umap = adata.obsm["X_umap"]
x_min, x_max = umap[:, 0].min(), umap[:, 0].max()
y_min, y_max = umap[:, 1].min(), umap[:, 1].max()

fig, axes = plt.subplots(2, len(disease_order), figsize=(1.5 * len(disease_order), 2.5), dpi=300,
                         sharex=True, sharey=True, gridspec_kw={"wspace": 0.1, "hspace": 0.3})

for i, disease in enumerate(disease_order):
    disease_adata = adata[adata.obs["Condition"] == disease]
    
    scatter1 = axes[0, i].scatter(disease_adata.obsm["X_umap"][:, 0], disease_adata.obsm["X_umap"][:, 1],
                                  c=disease_adata.obs["inflammatory_score"], cmap=cmap, s=0.02, alpha=1,
                                  vmin=vmin_inflammatory, vmax=vmax_inflammatory, edgecolor="none", rasterized=True)
                                  
    scatter2 = axes[1, i].scatter(disease_adata.obsm["X_umap"][:, 0], disease_adata.obsm["X_umap"][:, 1],
                                  c=disease_adata.obs["cytokine_score"], cmap=cmap, s=0.02, alpha=1,
                                  vmin=vmin_cytokine, vmax=vmax_cytokine, edgecolor="none", rasterized=True)
                                  
    axes[0, i].text(0.5, 1.15, disease_labels[i], transform=axes[0, i].transAxes,
                    ha="center", va="bottom", fontsize=6, fontweight="bold")

for ax in axes.flat:
    ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max))
    ax.set_facecolor("white")
    ax.grid(False)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[:, 0]:
    ax.spines["left"].set_visible(True)
    ax.spines["left"].set_linewidth(1)
    ax.set_ylabel("UMAP2", fontsize=6)

for ax in axes.flat:
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_linewidth(1)

for ax in axes[1, :]:
    ax.set_xlabel("UMAP1", fontsize=6)

fig.text(0.5, 1.0, "Inflammatory Score", ha="center", va="bottom", fontsize=6, fontweight="bold")
fig.text(0.5, 0.5, "Cytokine Score", ha="center", va="bottom", fontsize=6, fontweight="bold")

cbar1 = fig.colorbar(scatter1, ax=axes[0, :], location="right", shrink=0.6, aspect=20, pad=0.02)
cbar2 = fig.colorbar(scatter2, ax=axes[1, :], location="right", shrink=0.6, aspect=20, pad=0.02)

for cbar in [cbar1, cbar2]:
    cbar.ax.tick_params(labelsize=5)
    cbar.outline.set_visible(False)

plt.show()

In [ ]:
sc.tl.score_genes(adata, gene_list=['TLR4'], score_name='TLR4_score')
sc.tl.score_genes(adata, gene_list=['MYD88'], score_name='MYD88_score')

In [ ]:
sbp = adata[adata.obs["Condition"] == "SKPP"].obs
scores = sbp.groupby("celltype", observed=False)[["TLR4_score", "MYD88_score"]].sum().clip(lower=0)
percentages = scores.div(scores.sum(axis=0), axis=1).mul(100)

genes = ["TLR4_score", "MYD88_score"]
colors = ["#E64B35", "#4DBBD5"]
celltypes = percentages.index

fig, axes = plt.subplots(2, 1, figsize=(8, 6))

for ax, gene, color in zip(axes, genes, colors):
    values = percentages[gene]
    bars = ax.bar(range(len(celltypes)), values, color=color, alpha=0.8, linewidth=0.5)
    ax.set_title(gene.replace("_score", "_Score"), fontsize=14, fontweight="bold")
    ax.set_ylabel("Percentage (%)", fontsize=12, fontweight="bold")
    ax.set_xticks(range(len(celltypes)))
    ax.set_xlim(-0.5, len(celltypes) - 0.5)
    ax.set_ylim(0, values.max() * 1.15)
    ax.grid(False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(0.5)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + values.max() * 0.015,
                f"{value:.1f}%", ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=90)
axes[0].set_xticklabels([])
axes[1].set_xticklabels(celltypes, rotation=90, ha="center", fontsize=8)

plt.show()

<h1>CellphoneDB</h1>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
import ktplotspy as kpy
import os

In [ ]:
cpdb_file_path = '/home/xiaoquan/scanpy/KP/CellphoneDB/v5.0.0/cellphonedb.zip'
meta_file_path = '/home/xiaoquan/scanpy/KP/Results/2-Result2/cpdb_results/adata_SKPP_cell_annotations.csv'
counts_file_path = '/home/xiaoquan/scanpy/KP/Results/QC/adata_SKPP.h5ad'
out_path = 'cellphonedb_results_SKPP'
os.makedirs(out_path, exist_ok=True)

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path=cpdb_file_path,
    meta_file_path=meta_file_path,
    counts_file_path=counts_file_path,
    counts_data="hgnc_symbol",
    score_interactions=True,
    iterations=1000,
    threshold=0.1,
    threads=5,
    debug_seed=42,
    result_precision=3,
    pvalue=0.05,
    separator="|",
    debug=False,
    output_path=out_path
)

In [ ]:
sns.set()
with plt.style.context({'xtick.labelsize': 12, 'ytick.labelsize': 12}):
    g = kpy.plot_cpdb_heatmap(
        pvals = cpdb_results_SKPP['pvalues'],
        degs_analysis = False,
        figsize = (6, 6),
        title = "Sum of significant interactions",
        low_col = "#4DBBD5",
        high_col =  "#E64B35", 
        symmetrical = True,
        cbar_pos = (1, 0.3, 0.02, 0.4),
        linecolor = 'w',        
        annot = False,         
        linewidths = 0.5       
    )
plt.savefig('Fig.S6/cellphonedb_interactions2.pdf', dpi=300, bbox_inches='tight')
plt.show()